# 前馈神经网络（下）鸢尾花三分类

本节用 PyTorch 完成鸢尾花数据集的三分类，目标是把 chap2 的 `Runner` 升级到 **`RunnerV2`**：

- 集成 `DataLoader` 批量训练
- 在验证集上追踪 **accuracy** 等任意 metric
- 自动保存 dev metric 最优的 checkpoint

## 1. 数据集：sklearn 的 iris

150 个样本、4 个特征（花瓣/花萼的长宽）、3 个类别。

In [ ]:
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris

X, y = load_iris(return_X_y=True)
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)
print('X:', X.shape, 'y:', y.shape, 'classes:', y.unique().tolist())

### train / dev / test 划分 + 标准化

iris 是按类别排列的，洗牌后按 6:2:2 切。标准化的 mean/std 只在训练集上拟合。

In [ ]:
torch.manual_seed(0)
perm = torch.randperm(len(X))
n_train = int(0.6 * len(X))
n_dev   = int(0.2 * len(X))
tr, dv, te = perm[:n_train], perm[n_train:n_train+n_dev], perm[n_train+n_dev:]
X_train, y_train = X[tr], y[tr]
X_dev,   y_dev   = X[dv], y[dv]
X_test,  y_test  = X[te], y[te]

mean, std = X_train.mean(0), X_train.std(0)
X_train = (X_train - mean) / std
X_dev   = (X_dev   - mean) / std
X_test  = (X_test  - mean) / std

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=16, shuffle=True)
dev_loader   = DataLoader(TensorDataset(X_dev,   y_dev),   batch_size=32)
test_loader  = DataLoader(TensorDataset(X_test,  y_test),  batch_size=32)
print(f'train {len(X_train)}  dev {len(X_dev)}  test {len(X_test)}')

## 2. 模型：两层 MLP

用 `nn.Module` 子类的写法（比 `nn.Sequential` 更灵活，后面常用）。输出 3 维 logits，配 `CrossEntropyLoss`。

In [ ]:
class IrisMLP(nn.Module):
    def __init__(self, in_dim=4, hidden=16, n_class=3):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, n_class)

    def forward(self, x):
        h = F.relu(self.fc1(x))
        return self.fc2(h)        # 返回 logits

## 3. `RunnerV2`：best-model 追踪

在 chap2 `Runner` 基础上加：

- `metric_fn(logits, y) -> float`：返回一个标量；`higher_is_better=True` 时把最大值认作最优。
- 训练循环里实时与历史最佳比较，更优就把 `state_dict` 写到 `best_path`。

In [ ]:
class RunnerV2:
    def __init__(self, model, optimizer, loss_fn, metric_fn=None, higher_is_better=True):
        self.model = model
        self.optimizer = optimizer
        self.loss_fn = loss_fn
        self.metric_fn = metric_fn          # 评估时用；None 则用 loss
        self.higher_is_better = higher_is_better
        self.history = {'train_loss': [], 'dev_loss': [], 'dev_metric': []}

    def fit(self, train_loader, dev_loader, num_epochs=100, log_every=10, best_path=None):
        best = -float('inf') if self.higher_is_better else float('inf')
        for epoch in range(num_epochs):
            self.model.train()
            running, n = 0.0, 0
            for x, y in train_loader:
                self.optimizer.zero_grad()
                loss = self.loss_fn(self.model(x), y)
                loss.backward()
                self.optimizer.step()
                running += loss.item() * x.size(0); n += x.size(0)
            train_loss = running / n
            dev_loss, dev_metric = self._eval(dev_loader)
            self.history['train_loss'].append(train_loss)
            self.history['dev_loss'].append(dev_loss)
            self.history['dev_metric'].append(dev_metric)

            improved = (dev_metric > best) if self.higher_is_better else (dev_metric < best)
            if improved:
                best = dev_metric
                if best_path is not None:
                    Path(best_path).parent.mkdir(parents=True, exist_ok=True)
                    torch.save(self.model.state_dict(), best_path)
            if (epoch + 1) % log_every == 0:
                tag = ' *' if improved else ''
                print(f'epoch {epoch+1:4d}  train_loss={train_loss:.4f}  dev_loss={dev_loss:.4f}  dev_metric={dev_metric:.4f}{tag}')

    @torch.no_grad()
    def _eval(self, loader):
        self.model.eval()
        total_loss, total_m, n = 0.0, 0.0, 0
        for x, y in loader:
            out = self.model(x)
            total_loss += self.loss_fn(out, y).item() * x.size(0)
            if self.metric_fn is not None:
                total_m += self.metric_fn(out, y) * x.size(0)
            n += x.size(0)
        dev_loss = total_loss / n
        dev_metric = (total_m / n) if self.metric_fn is not None else dev_loss
        return dev_loss, dev_metric

    def evaluate(self, loader):
        return self._eval(loader)

    @torch.no_grad()
    def predict(self, x):
        self.model.eval()
        return self.model(x)

    def load(self, path):
        self.model.load_state_dict(torch.load(path))

## 4. 训练

In [ ]:
def accuracy(logits, y):
    return (logits.argmax(dim=1) == y).float().mean().item()

torch.manual_seed(0)
model = IrisMLP(in_dim=4, hidden=16, n_class=3)
runner = RunnerV2(
    model,
    optim.Adam(model.parameters(), lr=0.01),
    nn.CrossEntropyLoss(),
    metric_fn=accuracy,
    higher_is_better=True,
)
runner.fit(train_loader, dev_loader, num_epochs=150, log_every=20, best_path='./models/iris_best.pt')

## 5. 评估 best-model

训练过程中可能后期 dev 准确率下降——加载我们在训练时存下来的最佳 checkpoint，再在测试集上看效果。

In [ ]:
runner.load('./models/iris_best.pt')
test_loss, test_acc = runner.evaluate(test_loader)
print(f'best-model  test loss={test_loss:.4f}  test acc={test_acc:.4f}')

## 6. 训练曲线

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(runner.history['train_loss'], label='train')
ax1.plot(runner.history['dev_loss'],   label='dev')
ax1.set_xlabel('epoch'); ax1.set_ylabel('CE loss'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(runner.history['dev_metric'])
ax2.set_xlabel('epoch'); ax2.set_ylabel('dev accuracy'); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 7. 单点预测

In [ ]:
class_names = ['setosa', 'versicolor', 'virginica']
x0 = X_test[:1]
with torch.no_grad():
    logits = runner.predict(x0)
    probs  = F.softmax(logits, dim=1).squeeze().tolist()
print(f'true  : {class_names[y_test[0].item()]}')
print(f'pred  : {class_names[logits.argmax(dim=1).item()]}')
print(f'probs : ' + ', '.join(f"{n}={p:.3f}" for n, p in zip(class_names, probs)))

## 小结

- `nn.CrossEntropyLoss` 接受 logits + 整数标签，内部做 log-softmax + NLL；模型最后一层只输出 logits。
- **best-model 追踪** 是训练 ML 模型的标配。`RunnerV2` 在 `fit()` 里实时比较 dev metric 并写 checkpoint。
- 把数据标准化（mean/std 只在训练集上拟合）后，MLP 用 16 个隐藏单元就能拿到 ~97% 测试准确率。
- `DataLoader(shuffle=True)` 在每个 epoch 重新洗 batch；`eval` 阶段不需要 shuffle。